# wardowl paper, experiment X1: under-retrieval under permission filters

Why did Qdrant reach recall@10 0.561, and pgvector return nothing to ten board members, in the enterprise RAG acceptance run of 2026-10-24? This notebook runs the paper's first experiment with the scripts of [`benchmarks/paper`](https://github.com/muzakkirhussain011/wardowl/tree/paper/x1-underretrieval/benchmarks/paper) (see its README). The predictions and the rules that judge them were fixed beforehand in `PREREGISTRATION.md`.

1. **X0** re-analyses the original run's `answers.jsonl` offline (CPU only).
2. **X1, smoke**: a small corpus, one build, every setting and code path, then its statistics. If it fails, the notebook stops here.
3. **X1, full**: the pgvector grid (P-a to P-h), then the Qdrant grid (Q-a to Q-e), retrieval only, each build-time variant built three times. Every hit is re-checked by wardowl and by the scenario's oracle; any leak stops the run.
4. **Statistics**: cluster-bootstrap intervals, paired tests and the pre-registered verdicts.

**Notebook settings:** Accelerator **GPU T4 x2**, **Internet on**. **Input:** add the output of the original run (notebook `notebookd8377caca8`) with *Add Input*; X0 needs its `answers.jsonl` and `results.json`, and X1 reuses its cached embeddings (`cache/chunks-*.npy`) when they are there. Without them X0 fails (the rest still runs) and X1 embeds the corpus again (about ten minutes).

Outputs, in `/kaggle/working`: `x0/`, `x1-smoke/`, `x1/` (with `x1/stats/report.md`) and `cache/`. Each X1 setting is written as soon as it finishes, so a crash keeps what was done; a rerun of the X1 cells resumes.

In [ ]:
import time

SESSION_START = time.time()  # Kaggle stops a session 12 hours after it starts
BRANCH = "paper/x1-underretrieval"  # the scripts and the wardowl under test come from this branch
SRC = "/tmp/wardowl-src"  # noqa: S108 - the clone; its scripts run, the wardowl they import is installed
ORIGINAL = ""  # the original run's output folder; empty: found under /kaggle/input
HOURS = 11.25  # no X1 setting starts after this many hours of the session
PGVECTOR_HOURS = 5.5  # nor a pgvector setting after this many hours of its own cell, so Qdrant gets its turn

In [ ]:
!rm -rf {SRC} && git clone -q --branch {BRANCH} https://github.com/muzakkirhussain011/wardowl.git {SRC}
!git -C {SRC} log -1 --format="%H %cs %s"
!pip install -q "wardowl[pgvector,qdrant] @ git+https://github.com/muzakkirhussain011/wardowl@{BRANCH}" "qdrant-client>=1.16" "sentence-transformers>=3" "datasets>=2.19" psutil pyarrow scipy

In [ ]:
# PostgreSQL with pgvector 0.8.7, built from source as in the original run: Kaggle has no Docker. Qdrant 1.19.2 is
# started and stopped by X1 itself, with benchmarks/kaggle_stores.sh.
!PG_TUNE=1 bash {SRC}/benchmarks/kaggle_setup.sh

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

CODES: dict[str, int] = {}  # each step's exit code: 0 done, 1 done with failures, 3 a leak stopped it
DEADLINE = str(SESSION_START + HOURS * 3600)


def step(name: str, script: str, *args: str) -> int:
    """Runs one script of benchmarks/paper, printing its output as it goes, and returns its exit code."""
    command = [sys.executable, f"{SRC}/benchmarks/paper/{script}", *args]
    print("$", " ".join(command), flush=True)
    pipe = {"stdout": subprocess.PIPE, "stderr": subprocess.STDOUT, "text": True}
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(command, env=env, **pipe) as run:  # noqa: S603
        for line in run.stdout:
            print(line, end="", flush=True)
    CODES[name] = run.returncode
    print(f"{name}: exit code {run.returncode}", flush=True)
    return run.returncode


def show(path: str) -> None:
    page = Path(path)
    if page.exists():
        display(Markdown(page.read_text(encoding="utf-8")))
    else:
        print(f"{path} was not written: see the step's output above")

In [ ]:
import shutil

# A later session can continue an unfinished run: add the earlier version of this notebook as input too. Its x1/
# is copied here, and the X1 cells (--resume) skip every build it finished; a build it left half done is redone.
earlier = sorted(Path("/kaggle/input").glob("**/x1/manifest.json"))
if earlier and not Path("/kaggle/working/x1").exists():
    shutil.copytree(earlier[0].parent, "/kaggle/working/x1", ignore=shutil.ignore_patterns("stats"))
    print(f"continuing the X1 run in {earlier[0].parent}" + (f" (of {len(earlier)} found)" if len(earlier) > 1 else ""))
else:
    print("no earlier X1 run among the inputs: X1 starts afresh")

## X0: the original run's outputs, re-analysed (section 6.0)

In [ ]:
step("x0", "x0_reanalysis.py", *(["--input", ORIGINAL] if ORIGINAL else []))
show("/kaggle/working/x0/summary.md")

## X1, smoke: every setting and code path on a small corpus

About 3,000 chunks, a dozen people, one build of each variant, two timed passes. A failure here stops the notebook before the long run.

In [ ]:
smoke = step("x1 smoke", "x1_underretrieval.py", "--smoke", "--resume", "--deadline", DEADLINE)
smoke_stats = step("x1 smoke statistics", "x1_stats.py", "--in", "/kaggle/working/x1-smoke", "--resamples", "500")
show("/kaggle/working/x1-smoke/stats/report.md")
if smoke or smoke_stats:
    raise RuntimeError(f"the smoke run did not pass ({CODES}): see /kaggle/working/x1-smoke/manifest.json")

## X1: pgvector (section 6.2)

PostgreSQL + pgvector loaded as in the original run; the query-time grid P-a to P-h on three builds of the default index, and four settings on three builds of an m=32, ef_construction=200 index. EXPLAIN (ANALYZE, BUFFERS) on every short result and on 50 fixed questions.

In [ ]:
pgvector = ["--stores", "pgvector", "--resume", "--deadline", DEADLINE, "--max-hours", str(PGVECTOR_HOURS)]
step("x1 pgvector", "x1_underretrieval.py", *pgvector)

## X1: Qdrant (section 6.3)

Five layouts (as run, `is_tenant`, `is_tenant` with `m=0` and `payload_m=16`, a separate knowledge-base collection, one segment), three builds each; ACORN off, at 0.4 and at 1.0, `hnsw_ef` 64, 128 and 512, and exact search. Qdrant's search-path counters are read per person.

In [ ]:
step("x1 qdrant", "x1_underretrieval.py", "--stores", "qdrant", "--resume", "--deadline", DEADLINE)

## Statistics and the pre-registered verdicts (sections 6.5 and 6.6)

In [ ]:
step("x1 statistics", "x1_stats.py", "--in", "/kaggle/working/x1")
show("/kaggle/working/x1/stats/report.md")
unclean = {name: code for name, code in CODES.items() if code}
print("every step finished cleanly" if not unclean else f"steps that did not finish cleanly: {unclean}")
print("see the manifests in /kaggle/working/x0, x1-smoke and x1 for unsupported, skipped and failed settings")